# Siamese CNN with Strategic Dual-Anchor for Lung Nodule Spiculation Classification

## Novel Approach: Strategic Dual-Anchor Class Strategy

Instead of fixing one class as anchor, this model uses both classes as anchors strategically. This creates:

- Symmetric embedding space: both spiculated and non-spiculated act as reference points
- Balanced learning: each class clusters with itself while pushing away the other
- No class bias: neither class is privileged as anchor
- Robust feature space: morphology differences are learned bidirectionally
- Stable metrics: accuracy, sensitivity, and specificity reflect balanced performance

## Dataset
- Training: 43 spiculated + 43 non-spiculated images from selected/train
- Testing: 20 spiculated + 20 non-spiculated images from selected/test
- Image size: 71x71 pixels

```text
selected/train/
  spiculated/      -> triplet CSV anchor_idx/pos_idx/neg_idx point here
  non_spiculated/  -> triplet CSV anchor_idx/pos_idx/neg_idx point here

selected/test/
  spiculated/      -> plain image input for test evaluation only
  non_spiculated/  -> plain image input for test evaluation only
```

Metrics per epoch:
- Train phase: loss, accuracy, sensitivity, specificity
- Test phase: accuracy, sensitivity, specificity

Phases:
1. Train with triplet combinations (epoch-wise anchor-controlled sampling)
2. Test with simple image inputs (fixed selected/test set every epoch)

In [1]:
import gc
import os
import copy
import datetime
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg") # Headless server mode
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
#  CONFIGURATION
# =============================================================================
CPU_COUNT = os.cpu_count() or 8
CFG = {
    # Input paths (current workspace)
    "sp_selected_dir"   : r"/home/rpatil5/rohan/Siamese/2_rater/triplet/selected/train/spiculated",
    "nonsp_selected_dir": r"/home/rpatil5/rohan/Siamese/2_rater/triplet/selected/train/non_spiculated",
    "sp_test_dir"       : r"/home/rpatil5/rohan/Siamese/2_rater/triplet/selected/test/spiculated",
    "nonsp_test_dir"    : r"/home/rpatil5/rohan/Siamese/2_rater/triplet/selected/test/non_spiculated",

    # Semi-hard + hard triplets (combined)
    "sp_triplet_csv_semi"    : r"/home/rpatil5/rohan/Siamese/2_rater/triplet/experiments_triplet/1_semi_hard_alpha_0p2/SP_anchor.csv",
    "nonsp_triplet_csv_semi" : r"/home/rpatil5/rohan/Siamese/2_rater/triplet/experiments_triplet/1_semi_hard_alpha_0p2/NonSP_anchor.csv",
    "sp_triplet_csv_hard"    : r"/home/rpatil5/rohan/Siamese/2_rater/triplet/experiments_triplet/3_hard_delta_neg_0p2_to_0/SP_anchor.csv",
    "nonsp_triplet_csv_hard" : r"/home/rpatil5/rohan/Siamese/2_rater/triplet/experiments_triplet/3_hard_delta_neg_0p2_to_0/NonSP_anchor.csv",

    # Output (all training artifacts)
    "output_dir"        : r"/home/rpatil5/rohan/Siamese/2_rater/Siamese_Models_C/results/Siamese_archvSemiHard_Hard",

    # Multi-trial setup
    "n_trials"          : 20,
    "ci_z"              : 1.96,

    # Current split for CSV anchor indices (43 train images: 0..42)
    "train_anchor_max"  : 42,

    # Anchor control
    "anchor_max_occurrences": 75,

    # Model
    "img_size"          : 71,

    # Training
    "epochs"            : 20,
    "batch_size"        : 1024,
    "initial_lr"        : 0.0005,
    "weight_decay"      : 5e-3,
    "margin"            : 0.2,
    "distance_metric"   : "cosine", # "euclidean" or "cosine"
    "use_amp"           : torch.cuda.is_available(),
    "accumulation_steps": 1,
    "num_workers"       : 16,
    "pin_memory"        : True,
    "persistent_workers": True,
    "prefetch_factor"   : 6,
    "seed"              : 42,

    # Fast runtime flags
    "deterministic"     : False,
    "benchmark_cudnn"   : True,
    "channels_last"     : True,
    "use_torch_compile" : False,
    "knn_backend"       : "torch",

    # Train-test gap early stop
    "overfit_gap_threshold"  : 0.10,
    "overfit_gap_patience"   : 3,
    "test_eval_every"        : 1,
    "preload_eval_tensors"   : True,

    # KNN
    "k_neighbors"       : 9,

    # Device
    "device"            : "cuda:2" if torch.cuda.is_available() else "cpu",
}

print(CFG)

{'sp_selected_dir': '/home/rpatil5/rohan/Siamese/2_rater/triplet/selected/train/spiculated', 'nonsp_selected_dir': '/home/rpatil5/rohan/Siamese/2_rater/triplet/selected/train/non_spiculated', 'sp_test_dir': '/home/rpatil5/rohan/Siamese/2_rater/triplet/selected/test/spiculated', 'nonsp_test_dir': '/home/rpatil5/rohan/Siamese/2_rater/triplet/selected/test/non_spiculated', 'sp_triplet_csv_semi': '/home/rpatil5/rohan/Siamese/2_rater/triplet/experiments_triplet/1_semi_hard_alpha_0p2/SP_anchor.csv', 'nonsp_triplet_csv_semi': '/home/rpatil5/rohan/Siamese/2_rater/triplet/experiments_triplet/1_semi_hard_alpha_0p2/NonSP_anchor.csv', 'sp_triplet_csv_hard': '/home/rpatil5/rohan/Siamese/2_rater/triplet/experiments_triplet/3_hard_delta_neg_0p2_to_0/SP_anchor.csv', 'nonsp_triplet_csv_hard': '/home/rpatil5/rohan/Siamese/2_rater/triplet/experiments_triplet/3_hard_delta_neg_0p2_to_0/NonSP_anchor.csv', 'output_dir': '/home/rpatil5/rohan/Siamese/2_rater/Siamese_Models_C/results/Siamese_archvSemiHard_Hard'

In [3]:
#  UTILITIES
# =============================================================================

def log(msg: str):
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}]  {msg}", flush=True)

def set_seed(s: int, deterministic: bool = False):
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

In [4]:
# =============================================================================
class SCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 30, kernel_size=3)
        self.bn1   = nn.BatchNorm2d(30)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(30, 30, kernel_size=3)
        self.bn2   = nn.BatchNorm2d(30)
        self.pool2 = nn.MaxPool2d(2)

        self.dropout_conv = nn.Dropout2d(p=0.2) 
        self.dropout_fc   = nn.Dropout(p=0.4)   
        
        self.fc1   = nn.Linear(30 * 16 * 16, 128)

    def forward_once(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout_conv(x) 
        
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout_conv(x) 
        
        # flatten is safer than view for channels_last + torch.compile execution
        x = torch.flatten(x, 1)
        x = self.dropout_fc(x)   
        
        x = self.fc1(x)
        return F.normalize(x, p=2, dim=1)
    
    def forward(self, a, p, n):
        return self.forward_once(a), self.forward_once(p), self.forward_once(n)

In [5]:
# =============================================================================

class CombinedTripletDataset(Dataset):
    def __init__(self, df, sp_imgs, nsp_imgs, transform = None):
        self.df        = df.reset_index(drop=True)
        self.sp_imgs   = sp_imgs
        self.nsp_imgs  = nsp_imgs
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        source = row["source"]
        a_idx  = int(row["anchor_idx"])
        p_idx  = int(row["pos_idx"])
        n_idx  = int(row["neg_idx"])

        if source == "sp":
            anchor, positive, negative = (self.sp_imgs[a_idx], self.sp_imgs[p_idx], self.nsp_imgs[n_idx])
        else:
            anchor, positive, negative = (self.nsp_imgs[a_idx], self.nsp_imgs[p_idx], self.sp_imgs[n_idx])

        if self.transform:
            anchor   = self.transform(anchor)
            positive = self.transform(positive)
            negative = self.transform(negative)

        return anchor, positive, negative

In [6]:
def load_images_sorted(folder: str, img_size: int) -> np.ndarray:
    paths = sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_EXTS)
    imgs = []
    for p in paths:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            imgs.append(cv2.resize(img, (img_size, img_size)))
    arr = np.array(imgs, dtype=np.uint8)
    log(f"  Loaded {len(arr)} images  <-  {Path(folder).name}/")
    return arr


def build_anchor_control_epoch_df(base_df: pd.DataFrame, max_occ: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    sampled = []

    # Cap each (source, anchor_idx) to at most `max_occ` triplets per epoch.
    for _, grp in base_df.groupby(["source", "anchor_idx"], sort=False):
        keep = min(len(grp), max_occ)
        rs = int(rng.integers(0, 2_000_000_000))
        grp_sample = grp.sample(n=keep, replace=False, random_state=rs) if len(grp) > keep else grp.copy()
        sampled.append(grp_sample)

    epoch_df = pd.concat(sampled, ignore_index=True)
    epoch_df = epoch_df.sample(frac=1.0, random_state=int(rng.integers(0, 2_000_000_000))).reset_index(drop=True)
    return epoch_df


def extract_embeddings(model, imgs, device, batch_size: int = 256) -> np.ndarray:
    tf = T.Compose([T.ToPILImage(), T.ToTensor()])
    model.eval()
    all_emb = []
    with torch.no_grad():
        for i in range(0, len(imgs), batch_size):
            batch = torch.stack([tf(img) for img in imgs[i : i + batch_size]]).to(device)
            emb = model.forward_once(batch).cpu().numpy()
            all_emb.append(emb)
    return np.vstack(all_emb).astype(np.float32)


def extract_embeddings_torch(model, imgs, device, batch_size: int = 256) -> torch.Tensor:
    model.eval()
    all_emb = []

    if torch.is_tensor(imgs):
        data = imgs.to(device, non_blocking=True)
    else:
        data = images_to_tensor_batch(imgs, device, channels_last=CFG.get("channels_last", False))

    with torch.no_grad():
        for i in range(0, data.size(0), batch_size):
            batch = data[i : i + batch_size]
            emb = model.forward_once(batch)
            all_emb.append(emb)
    return torch.cat(all_emb, dim=0)


def images_to_tensor_batch(imgs: np.ndarray, device, channels_last: bool = False) -> torch.Tensor:
    # Convert uint8 [N,H,W] once to float tensor [N,1,H,W] on device.
    x = torch.as_tensor(imgs, dtype=torch.float32, device=device).unsqueeze(1) / 255.0
    if channels_last and device.type == "cuda":
        x = x.contiguous(memory_format=torch.channels_last)
    return x


def compute_classification_metrics(y_true, y_pred, prefix) -> dict:
    cm             = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    acc  = accuracy_score(y_true, y_pred)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        f"{prefix}_acc"  : round(acc,  4),
        f"{prefix}_sens" : round(sens, 4),
        f"{prefix}_spec" : round(spec, 4),
        f"{prefix}_tp"   : int(tp),
        f"{prefix}_tn"   : int(tn),
        f"{prefix}_fp"   : int(fp),
        f"{prefix}_fn"   : int(fn),
    }


def knn_evaluate(model, gallery_imgs, gallery_labels, query_imgs, query_labels, k, device, prefix) -> dict:
    if CFG.get("knn_backend", "torch") == "torch":
        gallery_emb = extract_embeddings_torch(model, gallery_imgs, device)
        query_emb = extract_embeddings_torch(model, query_imgs, device)

        g_labels = torch.as_tensor(gallery_labels, device=device, dtype=torch.long)
        q_labels = torch.as_tensor(query_labels, device=device, dtype=torch.long)

        if CFG.get("distance_metric") == "cosine":
            # Cosine similarity is maximized, so distance is 1 - similarity
            # Higher similarity -> smaller distance
            sim = F.cosine_similarity(query_emb.unsqueeze(1), gallery_emb.unsqueeze(0), dim=-1)
            dists = 1.0 - sim
        else: # Euclidean
            dists = torch.cdist(query_emb, gallery_emb, p=2)

        knn_idx = torch.topk(dists, k=k, dim=1, largest=False).indices
        knn_labels = g_labels[knn_idx]

        votes_pos = knn_labels.sum(dim=1)
        preds_t = (votes_pos * 2 >= k).long()

        preds = preds_t.detach().cpu().numpy()
        y_true = q_labels.detach().cpu().numpy()
        return compute_classification_metrics(y_true, preds, prefix)

    # Sklearn backend
    gallery_emb = extract_embeddings(model, gallery_imgs, device)
    query_emb   = extract_embeddings(model, query_imgs,   device)
    
    metric = "cosine" if CFG.get("distance_metric") == "cosine" else "euclidean"
    knn = KNeighborsClassifier(n_neighbors=k, metric=metric)
    
    knn.fit(gallery_emb, gallery_labels)
    preds = knn.predict(query_emb)
    return compute_classification_metrics(query_labels, preds, prefix)


def get_attention_boxes(model, img_tensor, device):
    """Pick one random feature-map channel per conv layer and draw one box per layer."""
    model.eval()
    activations = {}

    def get_activation(name):
        def hook(model, input, output):
            activations[name] = output.detach()
        return hook

    h1 = model.conv1.register_forward_hook(get_activation("conv1"))
    h2 = model.conv2.register_forward_hook(get_activation("conv2"))

    _ = model.forward_once(img_tensor.to(device))
    h1.remove()
    h2.remove()

    boxes = []

    def random_map_peak(feature_map):
        fmap = feature_map.detach().cpu()
        if fmap.dim() == 4:
            fmap = fmap.squeeze(0)
        rand_ch = int(np.random.randint(0, fmap.size(0)))
        _, _, _, max_loc = cv2.minMaxLoc(fmap[rand_ch].numpy())
        x, y = max_loc
        return x, y

    x1, y1 = random_map_peak(activations["conv1"])
    boxes.append({"color": "red", "x": x1 - 3, "y": y1 - 3, "w": 6, "h": 6})

    x2, y2 = random_map_peak(activations["conv2"])
    boxes.append({"color": "green", "x": x2 * 2 - 7, "y": y2 * 2 - 7, "w": 14, "h": 14})

    return boxes


def plot_ranked_grid(ranking_data, model, class_label, class_name, condition, condition_name, out_dir, device, top_n=12):
    filtered_data = [item for item in ranking_data if item["true_label"] == class_label and item["is_correct"] == condition]
    if not filtered_data:
        return

    top_images = filtered_data[: min(top_n, len(filtered_data))]
    fig, axes = plt.subplots(2, 6, figsize=(14, 5))
    k = CFG["k_neighbors"]
    plt.suptitle(f"Top {len(top_images)} Ranked {condition_name} {class_name}", fontsize=12, y=0.98, fontweight="bold")

    for i, ax in enumerate(axes.flat):
        if i < len(top_images):
            item = top_images[i]
            img = item["image"]
            ax.imshow(img, cmap="gray")

            if condition:
                title_str = f"R:{item['rank']} | N:{item['same_class_neighbors']}/{k}\nD:{item['avg_distance']:.3f}"
                title_color = "green"
            else:
                pred_lbl = "Spic" if item["predicted_label"] == 1 else "Non"
                title_str = f"R:{item['rank']} | N:{item['same_class_neighbors']}/{k}\nPred:{pred_lbl}"
                title_color = "red"

            ax.set_title(title_str, fontsize=8, color=title_color, fontweight="bold")
            ax.axis("off")

            t_img = T.ToTensor()(img).unsqueeze(0)
            for b in get_attention_boxes(model, t_img, device):
                ax.add_patch(patches.Rectangle((b["x"], b["y"]), b["w"], b["h"], lw=1, edgecolor=b["color"], facecolor="none"))
        else:
            ax.axis("off")

    plt.tight_layout()
    safe_name = f"{condition_name}_{class_name}".replace(" ", "_").replace("-", "_")
    plt.savefig(out_dir / f"{safe_name}.png", dpi=150)
    plt.close()

In [7]:
def _ci95(series: pd.Series, z: float = 1.96):
    arr = series.astype(float).values
    n = len(arr)
    mean = float(np.mean(arr)) if n else np.nan
    std = float(np.std(arr, ddof=1)) if n > 1 else 0.0
    se = std / np.sqrt(n) if n > 0 else np.nan
    margin = z * se if n > 0 else np.nan
    low = mean - margin if n > 0 else np.nan
    high = mean + margin if n > 0 else np.nan
    return mean, low, high, std


def main():
    device = torch.device(CFG["device"])
    amp_enabled = bool(CFG["use_amp"]) and (device.type == "cuda")
    amp_device = "cuda" if device.type == "cuda" else "cpu"
    out = Path(CFG["output_dir"])
    out.mkdir(parents=True, exist_ok=True)

    models_dir = out / "trial_models"
    metrics_dir = out / "trial_metrics"
    models_dir.mkdir(parents=True, exist_ok=True)
    metrics_dir.mkdir(parents=True, exist_ok=True)

    log("=" * 72)
    log("  SIAMESE NETWORK - 20-TRIAL SEMI+HARD TRAINING (TRAIN + TEST ONLY)")
    log(f"  Device  : {device}")
    log(f"  Trials  : {CFG['n_trials']}")
    log(f"  Anchor cap per epoch: {CFG['anchor_max_occurrences']}")
    log(f"  Gap stop: (train_acc - test_acc) > {CFG['overfit_gap_threshold']*100:.1f}% for {CFG['overfit_gap_patience']} consecutive epochs")
    log("=" * 72)

    # 1) LOAD IMAGE ARRAYS ONCE
    log("STEP 1 >> Loading images")
    sp_train_imgs = load_images_sorted(CFG["sp_selected_dir"], CFG["img_size"])
    nsp_train_imgs = load_images_sorted(CFG["nonsp_selected_dir"], CFG["img_size"])
    sp_test_imgs = load_images_sorted(CFG["sp_test_dir"], CFG["img_size"])[:20]
    nsp_test_imgs = load_images_sorted(CFG["nonsp_test_dir"], CFG["img_size"])[:20]

    # 2) LOAD TRIPLETS (TRAIN ONLY): combine semi-hard + hard
    sp_df_semi = pd.read_csv(CFG["sp_triplet_csv_semi"])
    nsp_df_semi = pd.read_csv(CFG["nonsp_triplet_csv_semi"])
    sp_df_hard = pd.read_csv(CFG["sp_triplet_csv_hard"])
    nsp_df_hard = pd.read_csv(CFG["nonsp_triplet_csv_hard"])
    sp_df = pd.concat([sp_df_semi, sp_df_hard], ignore_index=True)
    nsp_df = pd.concat([nsp_df_semi, nsp_df_hard], ignore_index=True)
    sp_df["source"] = "sp"
    nsp_df["source"] = "nsp"

    T_MAX = CFG["train_anchor_max"]
    sp_train_df = sp_df[(sp_df["anchor_idx"] <= T_MAX) & (sp_df["pos_idx"] <= T_MAX) & (sp_df["neg_idx"] <= T_MAX)].reset_index(drop=True)
    nsp_train_df = nsp_df[(nsp_df["anchor_idx"] <= T_MAX) & (nsp_df["pos_idx"] <= T_MAX) & (nsp_df["neg_idx"] <= T_MAX)].reset_index(drop=True)
    base_train_df = pd.concat([sp_train_df, nsp_train_df], ignore_index=True)

    # KNN gallery and query arrays
    knn_gallery_imgs = np.concatenate([nsp_train_imgs, sp_train_imgs])
    knn_gallery_labels = np.array([0] * len(nsp_train_imgs) + [1] * len(sp_train_imgs))

    train_query_imgs = knn_gallery_imgs
    train_query_labels = knn_gallery_labels

    test_query_imgs = np.concatenate([nsp_test_imgs, sp_test_imgs])
    test_query_labels = np.array([0] * len(nsp_test_imgs) + [1] * len(sp_test_imgs))

    # Option 4: preload evaluation inputs as GPU tensors once.
    if CFG.get("preload_eval_tensors", True):
        knn_gallery_eval_inputs = images_to_tensor_batch(
            knn_gallery_imgs, device, channels_last=CFG.get("channels_last", False)
        )
        train_query_eval_inputs = knn_gallery_eval_inputs
        test_query_eval_inputs = images_to_tensor_batch(
            test_query_imgs, device, channels_last=CFG.get("channels_last", False)
        )
    else:
        knn_gallery_eval_inputs = knn_gallery_imgs
        train_query_eval_inputs = train_query_imgs
        test_query_eval_inputs = test_query_imgs

    trial_rows = []

    for trial in range(1, CFG["n_trials"] + 1):
        trial_seed = CFG["seed"] + trial
        set_seed(trial_seed, deterministic=CFG['deterministic'])

        log("-" * 72)
        log(f"TRIAL {trial}/{CFG['n_trials']}  |  seed={trial_seed}")

        train_tf = T.Compose([
            T.ToPILImage(),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomRotation(degrees=90),
            T.ToTensor(),
        ])

        loader_kwargs = dict(
            batch_size=CFG["batch_size"],
            num_workers=CFG["num_workers"],
            pin_memory=CFG["pin_memory"],
            persistent_workers=CFG["persistent_workers"],
            prefetch_factor=CFG["prefetch_factor"],
        )

        first_epoch_train_df = build_anchor_control_epoch_df(
            base_train_df,
            max_occ=CFG["anchor_max_occurrences"],
            seed=trial_seed * 10_000 + 1,
        )
        train_drop_last = len(first_epoch_train_df) >= CFG["batch_size"]
        first_train_ds = CombinedTripletDataset(first_epoch_train_df, sp_train_imgs, nsp_train_imgs, train_tf)
        first_train_loader = DataLoader(first_train_ds, shuffle=True, drop_last=train_drop_last, **loader_kwargs)
        steps_per_epoch = max(1, len(first_train_loader))

        max_anchor_occ_epoch = int(first_epoch_train_df.groupby(["source", "anchor_idx"]).size().max())
        log(f"  Anchor-control train triplets/epoch: {len(first_epoch_train_df)} | max anchor usage in epoch: {max_anchor_occ_epoch}")

        model = SCNN().to(device)
        if CFG.get("channels_last", False) and device.type == "cuda":
            model = model.to(memory_format=torch.channels_last)
        if CFG.get("use_torch_compile", False) and hasattr(torch, "compile"):
            try:
                model = torch.compile(model, mode="max-autotune")
            except Exception as e:
                log(f"  torch.compile skipped: {e}")

        if CFG.get("distance_metric") == "cosine":
            criterion = nn.TripletMarginWithDistanceLoss(
                distance_function=lambda x, y: 1.0 - F.cosine_similarity(x, y),
                margin=CFG["margin"]
            )
        else: # Euclidean
            criterion = nn.TripletMarginLoss(margin=CFG["margin"])

        optimizer = optim.AdamW(model.parameters(), lr=CFG["initial_lr"], weight_decay=CFG["weight_decay"])
        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=CFG["initial_lr"],
            epochs=CFG["epochs"],
            steps_per_epoch=steps_per_epoch,
            pct_start=0.3,
            anneal_strategy="cos",
        )
        scaler = torch.amp.GradScaler(amp_device, enabled=amp_enabled)

        best_test_acc = -1.0
        best_test_sens = -1.0
        best_train_test_gap = float("inf")
        best_weights = None
        gap_overfit_count = 0
        all_metrics = []

        for epoch in range(1, CFG["epochs"] + 1):
            model.train()

            epoch_train_df = build_anchor_control_epoch_df(
                base_train_df,
                max_occ=CFG["anchor_max_occurrences"],
                seed=trial_seed * 10_000 + epoch,
            )
            train_drop_last = len(epoch_train_df) >= CFG["batch_size"]
            train_ds = CombinedTripletDataset(epoch_train_df, sp_train_imgs, nsp_train_imgs, train_tf)
            train_loader = DataLoader(train_ds, shuffle=True, drop_last=train_drop_last, **loader_kwargs)

            for step, (a, p, n) in enumerate(train_loader):
                a = a.to(device, non_blocking=True)
                p = p.to(device, non_blocking=True)
                n = n.to(device, non_blocking=True)
                if CFG.get("channels_last", False) and device.type == "cuda":
                    a = a.contiguous(memory_format=torch.channels_last)
                    p = p.contiguous(memory_format=torch.channels_last)
                    n = n.contiguous(memory_format=torch.channels_last)

                with torch.amp.autocast(amp_device, enabled=amp_enabled):
                    out_a, out_p, out_n = model(a, p, n)
                    loss = criterion(out_a, out_p, out_n) / CFG["accumulation_steps"]

                scaler.scale(loss).backward()

                if (step + 1) % CFG["accumulation_steps"] == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()



            tr = knn_evaluate(
                model,
                knn_gallery_eval_inputs,
                knn_gallery_labels,
                train_query_eval_inputs,
                train_query_labels,
                CFG["k_neighbors"],
                device,
                "train",
            )

            te = knn_evaluate(
                model,
                knn_gallery_eval_inputs,
                knn_gallery_labels,
                test_query_eval_inputs,
                test_query_labels,
                CFG["k_neighbors"],
                device,
                "test",
            )

            max_anchor_occ_epoch = int(epoch_train_df.groupby(["source", "anchor_idx"]).size().max())
            train_test_gap = float(tr["train_acc"] - te["test_acc"])
            log(
                f"T{trial:02d} E{epoch:02d} | "
                f"TrA={tr['train_acc']*100:5.1f}% TrSe={tr['train_sens']*100:5.1f}% TrSp={tr['train_spec']*100:5.1f}% | "
                f"TeA={te['test_acc']*100:5.1f}% TeSe={te['test_sens']*100:5.1f}% TeSp={te['test_spec']*100:5.1f}%"
            )

            all_metrics.append(
                {
                    "trial": trial,
                    "epoch": epoch,
                    "train_acc": tr["train_acc"],
                    "train_sens": tr["train_sens"],
                    "train_spec": tr["train_spec"],
                    "test_acc": te["test_acc"],
                    "test_sens": te["test_sens"],
                    "test_spec": te["test_spec"],
                }
            )

            # Save best model by test sensitivity, then test accuracy, then smaller train-test gap.
            if (
                (te["test_sens"] > best_test_sens)
                or (te["test_sens"] == best_test_sens and te["test_acc"] > best_test_acc)
                or (
                    te["test_sens"] == best_test_sens
                    and te["test_acc"] == best_test_acc
                    and abs(train_test_gap) < abs(best_train_test_gap)
                )
            ):
                best_test_sens = te["test_sens"]
                best_test_acc = te["test_acc"]
                best_train_test_gap = train_test_gap
                best_weights = copy.deepcopy(model.state_dict())

            if train_test_gap > CFG["overfit_gap_threshold"]:
                gap_overfit_count += 1
            else:
                gap_overfit_count = 0

            if gap_overfit_count >= CFG["overfit_gap_patience"]:
                log(
                    f"  Trial {trial}: early stop (train-test acc gap > {CFG['overfit_gap_threshold']*100:.1f}% "
                    f"for {CFG['overfit_gap_patience']} consecutive epochs)"
                )
                break

        metrics_df = pd.DataFrame(all_metrics)

        saved_epoch_idx = metrics_df["test_sens"].idxmax()
        tied = metrics_df[metrics_df["test_sens"] == metrics_df.loc[saved_epoch_idx, "test_sens"]]
        saved_epoch_idx = tied["test_acc"].idxmax()
        saved_epoch_row = metrics_df.loc[saved_epoch_idx]
        saved_epoch_num = int(saved_epoch_row["epoch"])

        log(
            f"  -> Model saved from epoch {saved_epoch_num} "
            f"(test_sens={saved_epoch_row['test_sens']:.4f}, test_acc={saved_epoch_row['test_acc']:.4f})"
        )

        if best_weights is not None:
            model.load_state_dict(best_weights)

        trial_model_path = models_dir / f"trial_{trial:02d}_best_model.pth"
        torch.save(
            {
                "trial": trial,
                "seed": trial_seed,
                "model_state_dict": best_weights,
                "cfg": CFG,
                "best_test_acc": best_test_acc,
                "best_test_sens": best_test_sens,
                "best_train_test_gap": best_train_test_gap,
                "epochs_trained": len(all_metrics),
                "architecture": "SCNN_with_dropout",
                "anchor_control": {"max_occurrences": CFG["anchor_max_occurrences"], "epoch_resampling": True},
            },
            trial_model_path,
        )

        trial_metrics_csv = metrics_dir / f"trial_{trial:02d}_training_metrics.csv"
        metrics_df.to_csv(trial_metrics_csv, index=False)

        last_row = metrics_df.iloc[-1]
        gap_abs = abs(float(last_row["train_acc"]) - float(last_row["test_acc"]))

        trial_rows.append(
            {
                "trial": trial,
                "seed": trial_seed,
                "epochs_trained": int(len(metrics_df)),
                "best_test_acc": float(best_test_acc),
                "best_test_sens": float(best_test_sens),
                "best_train_test_gap": float(best_train_test_gap),
                "final_train_acc": float(last_row["train_acc"]),
                "final_test_acc": float(last_row["test_acc"]),
                "overfit_gap_abs": float(gap_abs),
                "test_acc": float(saved_epoch_row["test_acc"]),
                "test_sens": float(saved_epoch_row["test_sens"]),
                "test_spec": float(saved_epoch_row["test_spec"]),
                "model_path": str(trial_model_path),
                "metrics_csv": str(trial_metrics_csv),
            }
        )

        del model, optimizer, scheduler, scaler, first_train_ds, first_train_loader, metrics_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    summary_df = pd.DataFrame(trial_rows)
    summary_df["selection_score"] = summary_df["test_acc"] - summary_df["overfit_gap_abs"]

    summary_df = summary_df.sort_values(
        by=["selection_score", "overfit_gap_abs", "best_test_sens", "test_acc"],
        ascending=[False, True, False, False],
    ).reset_index(drop=True)

    best_trial_row = summary_df.iloc[0].to_dict()
    best_trial_num = int(best_trial_row["trial"])
    best_ckpt = torch.load(best_trial_row["model_path"], map_location="cpu", weights_only=False)

    best_model_path = out / "best_model_low_overfit.pth"
    torch.save(best_ckpt, best_model_path)

    summary_csv = out / "trial_summary_20_runs.csv"
    summary_df.to_csv(summary_csv, index=False)

    ci_rows = []
    for metric in ["test_acc", "test_sens", "test_spec"]:
        mean, low, high, std = _ci95(summary_df[metric], z=CFG["ci_z"])
        ci_rows.append(
            {
                "metric": metric,
                "n_trials": int(len(summary_df)),
                "mean": mean,
                "std": std,
                "ci95_low": max(0.0, low),
                "ci95_high": min(1.0, high),
            }
        )

    ci_df = pd.DataFrame(ci_rows)
    ci_csv = out / "test_metrics_95ci.csv"
    ci_df.to_csv(ci_csv, index=False)

    best_trial_test_csv = out / "best_trial_test_result.csv"
    pd.DataFrame([best_trial_row]).to_csv(best_trial_test_csv, index=False)

    log("=" * 72)
    log("  20-TRIAL RUN COMPLETE")
    log("=" * 72)
    log(f"  Best trial            : {best_trial_num}")
    log(f"  Best model saved      : {best_model_path}")
    log(f"  Trial summary saved   : {summary_csv}")
    log(f"  95% CI saved          : {ci_csv}")
    log(f"  Best trial row saved  : {best_trial_test_csv}")

In [8]:
if __name__ == "__main__":
    main()

[12:38:04]  ========================================================================
[12:38:04]    SIAMESE NETWORK - 20-TRIAL SEMI+HARD TRAINING (TRAIN + TEST ONLY)
[12:38:04]    Device  : cuda:2
[12:38:04]    Trials  : 20
[12:38:04]    Anchor cap per epoch: 75
[12:38:04]    Gap stop: (train_acc - test_acc) > 10.0% for 3 consecutive epochs
[12:38:04]  ========================================================================
[12:38:04]  STEP 1 >> Loading images
[12:38:04]    Loaded 43 images  <-  spiculated/
[12:38:04]    Loaded 43 images  <-  non_spiculated/
[12:38:04]    Loaded 20 images  <-  spiculated/
[12:38:04]    Loaded 20 images  <-  non_spiculated/
[12:38:05]  ------------------------------------------------------------------------
[12:38:05]  TRIAL 1/20  |  seed=43
[12:38:05]    Anchor-control train triplets/epoch: 6450 | max anchor usage in epoch: 75
[12:38:09]  T01 E01 | TrA= 75.6% TrSe= 60.5% TrSp= 90.7% | TeA= 65.0% TeSe= 55.0% TeSp= 75.0%
[12:38:11]  T01 E02 | TrA= 80.2% T

In [9]:
# =============================================================================
# TEST ALL TRIAL MODELS - Load each saved model and evaluate on test set
# =============================================================================

device = torch.device(CFG["device"])
out = Path(CFG["output_dir"])
models_dir = out / "trial_models"
metrics_dir = out / "trial_metrics"

log("=" * 72)
log("  TESTING ALL TRIAL MODELS ON TEST SET")
log("=" * 72)

# 1. LOAD IMAGES (from selected/train and selected/test)
log("STEP 1 >> Loading images for testing")
sp_train_imgs = load_images_sorted(CFG["sp_selected_dir"], CFG["img_size"])
nsp_train_imgs = load_images_sorted(CFG["nonsp_selected_dir"], CFG["img_size"])
sp_test_imgs = load_images_sorted(CFG["sp_test_dir"], CFG["img_size"])[:20]
nsp_test_imgs = load_images_sorted(CFG["nonsp_test_dir"], CFG["img_size"])[:20]

# 2. PREPARE TEST AND GALLERY DATA
T_MAX = CFG["train_anchor_max"]
train_only_nsp = nsp_train_imgs[:T_MAX + 1]
train_only_sp = sp_train_imgs[:T_MAX + 1]

knn_gallery_imgs = np.concatenate([train_only_nsp, train_only_sp])
knn_gallery_labels = np.array([0] * len(train_only_nsp) + [1] * len(train_only_sp))

test_query_imgs = np.concatenate([nsp_test_imgs, sp_test_imgs])
test_query_labels = np.array([0] * len(nsp_test_imgs) + [1] * len(sp_test_imgs))

log(f"  Gallery size: {len(knn_gallery_imgs)} (for KNN)")
log(f"  Test size: {len(test_query_imgs)} ({len(nsp_test_imgs)} non-spic + {len(sp_test_imgs)} spic)")

# 3. ITERATE OVER ALL TRIAL MODELS
test_results = []

for trial in range(1, CFG["n_trials"] + 1):
    log("-" * 72)
    log(f"Testing Trial {trial}/{CFG['n_trials']}")

    # Load model checkpoint
    trial_model_path = models_dir / f"trial_{trial:02d}_best_model.pth"
    if not trial_model_path.exists():
        log(f"  WARNING: Model not found at {trial_model_path}, skipping...")
        continue

    checkpoint = torch.load(trial_model_path, map_location=device, weights_only=False)

    # Initialize model and load weights
    model = SCNN().to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Evaluate on test set using KNN
    test_metrics = knn_evaluate(
        model,
        knn_gallery_imgs,
        knn_gallery_labels,
        test_query_imgs,
        test_query_labels,
        CFG["k_neighbors"],
        device,
        "test",
    )

    log(f"  Trial {trial:02d} Test Results: "
        f"Acc={test_metrics['test_acc']*100:5.1f}% | "
        f"Sens={test_metrics['test_sens']*100:5.1f}% | "
        f"Spec={test_metrics['test_spec']*100:5.1f}%")

    # Read existing training metrics
    trial_metrics_csv = metrics_dir / f"trial_{trial:02d}_training_metrics.csv"

    if trial_metrics_csv.exists():
        existing_df = pd.read_csv(trial_metrics_csv)

        for col, val in test_metrics.items():
            existing_df[col] = val

        # Save updated metrics
        existing_df.to_csv(trial_metrics_csv, index=False)
        log(f"  ✓ Saved test results to {trial_metrics_csv.name}")
    else:
        log(f"  WARNING: Metrics file not found at {trial_metrics_csv}, skipping...")

    # Store for summary
    test_results.append({
        "trial": trial,
        **test_metrics
    })

    # Clean up
    del model, checkpoint
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 4. SAVE SUMMARY OF ALL TEST RESULTS
test_summary_df = pd.DataFrame(test_results)
test_summary_path = out / "all_trials_test_results.csv"
test_summary_df.to_csv(test_summary_path, index=False)

log("=" * 72)
log("  ALL TRIAL TESTING COMPLETE")
log(f"  Summary saved to: {test_summary_path}")
log("=" * 72)

# Display summary statistics
log("\nTest Results Summary (across all trials):")
log(f"  Test Accuracy:    {test_summary_df['test_acc'].mean()*100:.2f}% ± {test_summary_df['test_acc'].std()*100:.2f}%")
log(f"  Test Sensitivity: {test_summary_df['test_sens'].mean()*100:.2f}% ± {test_summary_df['test_sens'].std()*100:.2f}%")
log(f"  Test Specificity: {test_summary_df['test_spec'].mean()*100:.2f}% ± {test_summary_df['test_spec'].std()*100:.2f}%")

test_summary_df

[12:46:10]  ========================================================================
[12:46:10]    TESTING ALL TRIAL MODELS ON TEST SET
[12:46:10]  ========================================================================
[12:46:10]  STEP 1 >> Loading images for testing
[12:46:10]    Loaded 43 images  <-  spiculated/
[12:46:11]    Loaded 43 images  <-  non_spiculated/
[12:46:11]    Loaded 20 images  <-  spiculated/
[12:46:11]    Loaded 20 images  <-  non_spiculated/
[12:46:11]    Gallery size: 86 (for KNN)
[12:46:11]    Test size: 40 (20 non-spic + 20 spic)
[12:46:11]  ------------------------------------------------------------------------
[12:46:11]  Testing Trial 1/20
[12:46:11]    Trial 01 Test Results: Acc= 75.0% | Sens= 80.0% | Spec= 70.0%
[12:46:11]    ✓ Saved test results to trial_01_training_metrics.csv
[12:46:11]  ------------------------------------------------------------------------
[12:46:11]  Testing Trial 2/20
[12:46:11]    Trial 02 Test Results: Acc= 70.0% | Sens= 80.0%

,trial,test_acc,test_sens,test_spec,test_tp,test_tn,test_fp,test_fn
0,1,0.750,0.80,0.70,16,14,6,4
1,2,0.700,0.80,0.60,16,12,8,4
2,3,0.800,0.90,0.70,18,14,6,2
3,4,0.725,0.75,0.70,15,14,6,5
4,5,0.700,0.75,0.65,15,13,7,5
5,6,0.825,0.90,0.75,18,15,5,2
6,7,0.800,0.90,0.70,18,14,6,2
7,8,0.750,0.80,0.70,16,14,6,4
8,9,0.725,0.80,0.65,16,13,7,4
9,10,0.700,0.70,0.70,14,14,6,6


In [10]:
# =============================================================================
# COMPUTE 95% CI FOR TRAINING AND TESTING METRICS (NO VALIDATION)
# =============================================================================
# Strategy: select per-trial epoch with highest test sensitivity and test accuracy tie-break.

log("=" * 72)
log("  COMPUTING 95% CI FOR TRAIN/TEST METRICS")
log("=" * 72)

metrics_dir = out / "trial_metrics"
combined_metrics = []

for trial in range(1, CFG["n_trials"] + 1):
    trial_metrics_csv = metrics_dir / f"trial_{trial:02d}_training_metrics.csv"

    if not trial_metrics_csv.exists():
        log(f"  WARNING: Metrics file not found for trial {trial}, skipping...")
        continue

    df = pd.read_csv(trial_metrics_csv)

    max_test_sens = df["test_sens"].max()
    best_sens_epochs = df[df["test_sens"] == max_test_sens]
    best_epoch_idx = best_sens_epochs["test_acc"].idxmax()
    best_epoch_row = df.loc[best_epoch_idx]

    metrics_record = {
        "trial": trial,
        "epoch": int(best_epoch_row["epoch"]),
        "train_acc": float(best_epoch_row["train_acc"]),
        "train_sens": float(best_epoch_row["train_sens"]),
        "train_spec": float(best_epoch_row["train_spec"]),
        "test_acc": float(best_epoch_row["test_acc"]),
        "test_sens": float(best_epoch_row["test_sens"]),
        "test_spec": float(best_epoch_row["test_spec"]),
    }

    combined_metrics.append(metrics_record)
    log(
        f"  Trial {trial:02d}: Selected epoch {int(best_epoch_row['epoch'])} "
        f"(test_sens={best_epoch_row['test_sens']:.4f}, test_acc={best_epoch_row['test_acc']:.4f})"
    )

combined_df = pd.DataFrame(combined_metrics)

log("\n" + "=" * 72)
log("  CALCULATING 95% CONFIDENCE INTERVALS")
log("=" * 72)

stats_dict = {}
for phase_prefix in ["train", "test"]:
    for metric_suffix in ["acc", "sens", "spec"]:
        col_name = f"{phase_prefix}_{metric_suffix}"
        mean, low, high, std = _ci95(combined_df[col_name], z=CFG["ci_z"])
        stats_dict[col_name] = {
            "mean": mean,
            "std": std,
            "ci_low": max(0.0, low),
            "ci_high": min(1.0, high),
        }

result_rows = []
for metric_name in ["Accuracy", "Sensitivity", "Specificity"]:
    metric_suffix = {"Accuracy": "acc", "Sensitivity": "sens", "Specificity": "spec"}[metric_name]

    row = {"Metric": metric_name}
    for phase_name, phase_prefix in [("Training", "train"), ("Testing", "test")]:
        col_name = f"{phase_prefix}_{metric_suffix}"
        s = stats_dict[col_name]
        row[phase_name] = f"{s['mean']*100:.2f} +/- {s['std']*100:.2f}"
        row[f"{phase_name}_CI_95%"] = f"[{s['ci_low']*100:.2f}, {s['ci_high']*100:.2f}]"

    result_rows.append(row)

ci_formatted_df = pd.DataFrame(result_rows)

ci_results = []
for phase_name, phase_prefix in [("Training", "train"), ("Testing", "test")]:
    for metric_name, metric_suffix in [("Accuracy", "acc"), ("Sensitivity", "sens"), ("Specificity", "spec")]:
        col_name = f"{phase_prefix}_{metric_suffix}"
        s = stats_dict[col_name]
        ci_results.append({
            "phase": phase_name,
            "metric": metric_name,
            "mean": s["mean"],
            "std": s["std"],
            "ci95_low": s["ci_low"],
            "ci95_high": s["ci_high"],
            "n_trials": len(combined_df),
        })

ci_detailed_df = pd.DataFrame(ci_results)

combined_metrics_path = out / "combined_train_test_metrics.csv"
combined_df.to_csv(combined_metrics_path, index=False)

ci_formatted_path = out / "combined_train_test_metrics_95ci_formatted.csv"
ci_formatted_df.to_csv(ci_formatted_path, index=False)

ci_detailed_path = out / "combined_train_test_metrics_95ci_detailed.csv"
ci_detailed_df.to_csv(ci_detailed_path, index=False)

log("\n" + "=" * 72)
log("  FILES SAVED")
log("=" * 72)
log(f"  Combined metrics (all trials): {combined_metrics_path}")
log(f"  95% CI formatted table: {ci_formatted_path}")
log(f"  95% CI detailed: {ci_detailed_path}")
log("=" * 72)

print("\n" + "=" * 90)
print("95% CONFIDENCE INTERVALS (Mean +/- Std) - All values in %")
print("=" * 90)
ci_formatted_df

[12:46:18]  ========================================================================
[12:46:18]    COMPUTING 95% CI FOR TRAIN/TEST METRICS
[12:46:18]  ========================================================================
[12:46:18]    Trial 01: Selected epoch 1 (test_sens=0.8000, test_acc=0.7500)
[12:46:18]    Trial 02: Selected epoch 1 (test_sens=0.8000, test_acc=0.7000)
[12:46:18]    Trial 03: Selected epoch 1 (test_sens=0.9000, test_acc=0.8000)
[12:46:18]    Trial 04: Selected epoch 1 (test_sens=0.7500, test_acc=0.7250)
[12:46:18]    Trial 05: Selected epoch 1 (test_sens=0.7500, test_acc=0.7000)
[12:46:18]    Trial 06: Selected epoch 1 (test_sens=0.9000, test_acc=0.8250)
[12:46:18]    Trial 07: Selected epoch 1 (test_sens=0.9000, test_acc=0.8000)
[12:46:18]    Trial 08: Selected epoch 1 (test_sens=0.8000, test_acc=0.7500)
[12:46:18]    Trial 09: Selected epoch 1 (test_sens=0.8000, test_acc=0.7250)
[12:46:18]    Trial 10: Selected epoch 1 (test_sens=0.7000, test_acc=0.7000)
[12:46

,Metric,Training,Training_CI_95%,Testing,Testing_CI_95%
0,Accuracy,78.55 +/- 2.71,"[77.36, 79.73]",73.12 +/- 4.65,"[71.09, 75.16]"
1,Sensitivity,69.42 +/- 5.95,"[66.81, 72.03]",76.75 +/- 8.78,"[72.90, 80.60]"
2,Specificity,87.68 +/- 3.30,"[86.23, 89.12]",69.50 +/- 4.84,"[67.38, 71.62]"


In [11]:
# =============================================================================
# VISUALIZE ALL TEST IMAGES WITH PREDICTIONS + CONV ATTENTION BOXES
# =============================================================================

log("=" * 72)
log("  VISUALIZING ALL TEST IMAGES WITH PREDICTIONS")
log("=" * 72)

device = torch.device(CFG["device"])
out = Path(CFG["output_dir"]); out.mkdir(parents=True, exist_ok=True)
models_dir = out / "trial_models"

# 1) Load train and test images used by this notebook setup
sp_train_imgs = load_images_sorted(CFG["sp_selected_dir"], CFG["img_size"])
nsp_train_imgs = load_images_sorted(CFG["nonsp_selected_dir"], CFG["img_size"])
sp_test_imgs = load_images_sorted(CFG["sp_test_dir"], CFG["img_size"])[:20]
nsp_test_imgs = load_images_sorted(CFG["nonsp_test_dir"], CFG["img_size"])[:20]

T_MAX = CFG["train_anchor_max"]
train_only_nsp = nsp_train_imgs[:T_MAX + 1]
train_only_sp = sp_train_imgs[:T_MAX + 1]

knn_gallery_imgs = np.concatenate([train_only_nsp, train_only_sp])
knn_gallery_labels = np.array([0] * len(train_only_nsp) + [1] * len(train_only_sp))

test_imgs = np.concatenate([nsp_test_imgs, sp_test_imgs])
test_labels = np.array([0] * len(nsp_test_imgs) + [1] * len(sp_test_imgs))

# 2) Load model checkpoint for prediction
best_model_path = out / "best_model_low_overfit.pth"
if best_model_path.exists():
    model_path = best_model_path
else:
    trial_ckpts = sorted(models_dir.glob("trial_*_best_model.pth"))
    if not trial_ckpts:
        raise FileNotFoundError(f"No model checkpoint found in {models_dir}")
    model_path = trial_ckpts[0]

checkpoint = torch.load(model_path, map_location=device, weights_only=False)
model = SCNN().to(device)
state_dict = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
model.load_state_dict(state_dict)
model.eval()

# 3) Predict test labels via KNN in embedding space
k = min(CFG["k_neighbors"], len(knn_gallery_labels))
if CFG.get("knn_backend", "torch") == "torch":
    gallery_emb = extract_embeddings_torch(model, knn_gallery_imgs, device)
    query_emb = extract_embeddings_torch(model, test_imgs, device)
    g_labels = torch.as_tensor(knn_gallery_labels, device=device, dtype=torch.long)
    
    if CFG.get("distance_metric") == "cosine":
        sim = F.cosine_similarity(query_emb.unsqueeze(1), gallery_emb.unsqueeze(0), dim=-1)
        dists = 1.0 - sim
    else: # Euclidean
        dists = torch.cdist(query_emb, gallery_emb, p=2)
        
    knn_idx = torch.topk(dists, k=k, dim=1, largest=False).indices
    knn_labels = g_labels[knn_idx]
    votes_pos = knn_labels.sum(dim=1)
    preds = (votes_pos * 2 >= k).long().detach().cpu().numpy()
else:
    gallery_emb = extract_embeddings(model, knn_gallery_imgs, device)
    query_emb = extract_embeddings(model, test_imgs, device)
    metric = "cosine" if CFG.get("distance_metric") == "cosine" else "euclidean"
    knn = KNeighborsClassifier(n_neighbors=k, metric=metric)
    knn.fit(gallery_emb, knn_gallery_labels)
    preds = knn.predict(query_emb)

acc = (preds == test_labels).mean()
log(f"  Prediction accuracy on visualization set: {acc*100:.2f}%")

# 4) Plot with predictions + conv attention boxes (red=conv1, green=conv2)
n_total = len(test_imgs)
n_cols = 8
n_rows = int(np.ceil(n_total / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 2.2))
axes = np.array(axes).reshape(-1)

for i in range(n_total):
    img = test_imgs[i]
    true_lbl = int(test_labels[i])
    pred_lbl = int(preds[i])
    is_correct = (true_lbl == pred_lbl)

    true_name = "Spic" if true_lbl == 1 else "Non"
    pred_name = "Spic" if pred_lbl == 1 else "Non"

    ax = axes[i]
    ax.imshow(img, cmap="gray")
    ax.set_title(f"T:{true_name} P:{pred_name}", fontsize=8, color=("green" if is_correct else "red"), fontweight="bold")
    ax.axis("off")

    t_img = T.ToTensor()(img).unsqueeze(0)
    for b in get_attention_boxes(model, t_img, device):
        ax.add_patch(patches.Rectangle((b["x"], b["y"]), b["w"], b["h"], lw=1.2, edgecolor=b["color"], facecolor="none"))

for i in range(n_total, len(axes)):
    axes[i].axis("off")

plt.suptitle("Test Images with Predictions and Conv Attention Boxes", fontsize=14, fontweight="bold", y=0.995)
plt.tight_layout()
save_path = Path(CFG["output_dir"]) / "all_test_images_with_predictions_boxes.png"
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

log(f"  Saved visualization to: {save_path}")
log("  ✓ Visualization complete")
log("=" * 72)

[12:46:18]  ========================================================================
[12:46:18]    VISUALIZING ALL TEST IMAGES WITH PREDICTIONS
[12:46:18]  ========================================================================
[12:46:18]    Loaded 43 images  <-  spiculated/
[12:46:18]    Loaded 43 images  <-  non_spiculated/
[12:46:18]    Loaded 20 images  <-  spiculated/
[12:46:18]    Loaded 20 images  <-  non_spiculated/
[12:46:19]    Prediction accuracy on visualization set: 82.50%
[12:46:21]    Saved visualization to: /home/rpatil5/rohan/Siamese/2_rater/Siamese_Models_C/results/Siamese_archvSemiHard_Hard/all_test_images_with_predictions_boxes.png
[12:46:21]    ✓ Visualization complete
[12:46:21]  ========================================================================
